# 📊 Exploration autonome

> ✍️ **Notebook dirigé** · complétez les cellules marquées ✍️ À VOUS.

**Jour 1 · 9h15** · 30 minutes, sans consignes techniques

*Analyse exploratoire et interprétation métier*

**Objectifs**
- Faire un diagnostic rapide de votre jeu de données
- Identifier les variables clés pour le commanditaire
- Formuler trois premières hypothèses

**Maison Lumen — Soualiho KAMAGATÉ et Julien FOURNIER**

## Le principe
Pendant 30 minutes, vous explorez **comme vous voulez**. Aucune consigne technique : seulement des questions auxquelles répondre à la fin.

1. Que représente une ligne ?
2. Quelle période, combien de clients, combien de lignes ?
3. Quelles variables comptent pour le commanditaire de votre brief ?
4. Quels problèmes de qualité voyez-vous ?
5. Quelles **trois hypothèses** voulez-vous tester aujourd'hui ?

In [1]:
import os, sys, subprocess, importlib.util
for paquet in ["pandas", "numpy", "matplotlib", "seaborn", "scipy", "statsmodels", "fairlearn", "gdown"]:
    if importlib.util.find_spec(paquet) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",paquet])
CONFIG_ID="1bW5RsToUHc-7pGiyHS_1B2OYkAFsrv3r"
if not os.path.exists("config_projet.py"):
    import gdown
    gdown.download(id=CONFIG_ID,output="config_projet.py",quiet=True)
print("✅ Configuration prête")

In [2]:
# Choisissez votre secteur : "commerce", "banque" ou "hotellerie"
SECTEUR = "commerce"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config_projet import charger, reference, nettoyage_minimal

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
brut, cfg = charger(SECTEUR)
print(cfg["nom"], "·", brut.shape[0], "lignes,", brut.shape[1], "colonnes")
from pathlib import Path
from IPython.display import display, Markdown
evenement=pd.Timestamp(cfg["evenement"][0])
sensibles=cfg["sensibles"]


Maison Lumen · 24224 lignes, 16 colonnes


In [3]:
from config_projet import brief
display(brief(SECTEUR))

# Brief · Maison Lumen

> Entreprise et données fictives, conçues pour la formation.

## Le commanditaire
**Camille Roux, directrice marketing et e-commerce.** Maison Lumen vend luminaires, textile, mobilier, décoration et cuisine dans 38 magasins, sur son site et, depuis peu, sur une application.

## Le contexte
- Le 15 avril 2024, le site a été refondu et l'application lancée. La direction veut savoir ce que cela a vraiment changé.
- Depuis 2023, un algorithme de propension choisit chaque semaine les clients qui reçoivent une **offre personnalisée**. Le service client remonte des plaintes de clients âgés qui « ne reçoivent jamais rien ».
- Le comité de direction arbitre en janvier le budget marketing de l'année.

## La demande
1. Comment évolue le chiffre d'affaires ? Quelle saisonnalité, quel effet de la refonte d'avril 2024 ?
2. Quels canaux, catégories et profils de clients portent la croissance ?
3. L'algorithme d'offres personnalisées traite-t-il tous les clients de façon équitable ? Notre clientèle est-elle représentative ?
4. Que recommandez-vous pour 2026, avec quels indicateurs de suivi, y compris environnementaux et sociaux ?

## Les données · `commerce_maison_lumen.csv`
Une ligne = une commande, de janvier 2023 à décembre 2025.

| Colonne | Description |
|---|---|
| `date_commande` | Date de la commande |
| `commande_id`, `client_id` | Identifiants |
| `genre`, `age`, `tranche_age` | Profil déclaré à l'inscription au programme client |
| `region` | Zone de résidence du client |
| `canal` | `magasin`, `web` ou `app` |
| `categorie` | Famille de produits |
| `montant_eur` | Montant TTC payé, remise déduite (négatif pour un avoir) |
| `remise_pct` | Remise appliquée, en % |
| `carte_fidelite` | 1 si le client a la carte de fidélité |
| `note_satisfaction` | Note de 1 à 5 laissée après l'achat (facultative) |
| `retour` | 1 si au moins un article a été retourné |
| `offre_personnalisee` | 1 si l'algorithme a envoyé une offre au client dans le mois précédent |
| `rachat_90j` | 1 si le client a racheté dans les 90 jours |

`reference_population.csv` donne la répartition de la population française adulte (approximation pédagogique inspirée de l'INSEE).


In [4]:
df=nettoyage_minimal(brut,cfg)
df["region"]=df.region.fillna("inconnu")
df["apres"]=df.date_commande.ge(evenement).astype(int)
df["periode"]=np.where(df.apres.eq(1),"Après refonte","Avant refonte")
print(len(brut),"→",len(df),"; lignes retirées :",len(brut)-len(df))
print("Doublons exacts :",brut.duplicated().sum(),"; avoirs négatifs après dédoublonnage :",brut.drop_duplicates().montant_eur.lt(0).sum())
display(brut.head())
display(brut.dtypes.to_frame("Type"))
display(brut.isna().sum().to_frame("Manquants"))
display(brut.describe(include="all").T)
print("Période :", df.date_commande.min().date(), "à", df.date_commande.max().date())
print("Commandes distinctes brutes :", brut.commande_id.nunique())
print("Identifiants répétés après nettoyage :", df.commande_id.duplicated().sum())
display(df.groupby("client_id")[sensibles].nunique().max().to_frame("Modalités maximum par client"))
display(Markdown(f"Le fichier brut contient **{len(brut):,} lignes** et **{brut.shape[1]} colonnes**. Après nettoyage, je travaille sur **{len(df):,} commandes** de **{df.client_id.nunique():,} clients**. Les données vont du **{df.date_commande.min():%d/%m/%Y} au {df.date_commande.max():%d/%m/%Y}**. Un client peut commander plusieurs fois : le nombre de commandes ne correspond pas au nombre de personnes."))

,date_commande,commande_id,client_id,genre,age,tranche_age,region,canal,categorie,montant_eur,remise_pct,carte_fidelite,note_satisfaction,retour,offre_personnalisee,rachat_90j
0,2023-01-01,CMD000000,ML04162,H,35,35-44,Île-de-France,magasin,Luminaire,97.01,20,0,NaN,0,1,1
1,2023-01-01,CMD000016,ML01192,F,51,45-54,Nord-Est,web,Textile,51.90,20,0,2.0,1,1,0
2,2023-01-01,CMD000015,ML00594,F,25,25-34,Ouest,app,Cuisine,40.73,20,1,4.0,0,0,0
3,2023-01-01,CMD000014,ML08607,H,29,25-34,Nord-Est,web,Textile,48.74,10,1,3.0,0,1,0
4,2023-01-01,CMD000013,ML00599,H,20,18-24,Sud-Ouest,web,Cuisine,35.60,30,1,4.0,0,0,0


,Type
date_commande,datetime64[ns]
commande_id,object
client_id,object
genre,object
age,int64
tranche_age,object
region,object
canal,object
categorie,object
montant_eur,float64


,Manquants
date_commande,0
commande_id,0
client_id,0
genre,0
age,0
tranche_age,0
region,241
canal,0
categorie,0
montant_eur,0


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
date_commande,24224,NaN,NaN,NaN,2024-08-18 14:06:47.793923072,2023-01-01 00:00:00,2023-12-06 00:00:00,2024-09-19 00:00:00,2025-05-10 06:00:00,2025-12-31 00:00:00,NaN
commande_id,24224,24103,CMD005964,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
client_id,24224,6768,ML07240,34,NaN,NaN,NaN,NaN,NaN,NaN,NaN
genre,24224,2,F,15197,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,24224.0,NaN,NaN,NaN,39.581737,18.0,28.0,37.0,49.0,85.0,14.621194
tranche_age,24224,6,25-34,7168,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,23983,6,Île-de-France,7612,NaN,NaN,NaN,NaN,NaN,NaN,NaN
canal,24224,3,magasin,12127,NaN,NaN,NaN,NaN,NaN,NaN,NaN
categorie,24224,5,Déco,7192,NaN,NaN,NaN,NaN,NaN,NaN,NaN
montant_eur,24224.0,NaN,NaN,NaN,77.659685,-128.31,34.6775,48.97,75.3225,5406.73,119.644847


,Modalités maximum par client
tranche_age,1
genre,1
region,2


Le fichier brut contient **24,224 lignes** et **16 colonnes**. Après nettoyage, je travaille sur **24,089 commandes** de **6,768 clients**. Les données vont du **01/01/2023 au 31/12/2025**. Un client peut commander plusieurs fois : le nombre de commandes ne correspond pas au nombre de personnes.

24224 → 24089 ; lignes retirées : 135
Doublons exacts : 121 ; avoirs négatifs après dédoublonnage : 14
Période : 2023-01-01 à 2025-12-31
Commandes distinctes brutes : 24103
Identifiants répétés après nettoyage : 0


## Vos réponses
1. **Une ligne = une commande**, pas un client unique.
2. **Période et volumes :** du 1er janvier 2023 au 31 décembre 2025 ; 24 224 lignes brutes, 24 089 commandes retenues et 6 768 clients uniques.
3. **Variables clés :** montant et date pour l’activité ; canal et catégorie pour comprendre les ventes ; offre personnalisée pour le ciblage ; rachat à 90 jours pour le comportement après achat ; âge, genre et région pour les écarts entre groupes.
4. **Qualité :** 121 doublons exacts et 14 avoirs négatifs après dédoublonnage. Les avoirs sont exclus selon le cours : le total n’est donc pas un CA comptable net des avoirs. La satisfaction est facultative ; je ne remplace pas ses valeurs absentes par zéro. La présence de l’app avant le lancement annoncé est un point à clarifier.
5. **Hypothèses :**
**H1.** Le CA mensuel est plus élevé après la refonte du site du 15 avril 2024. La saisonnalité sera prise en compte dans l’analyse temporelle.

**H2.** La hausse du CA mensuel est plus forte sur l’application que sur les autres canaux.

**H3.** Le taux d’offre personnalisée varie selon l’âge. La comparaison sera également menée à canal égal et avant/après la refonte.



💾 **Versionnez** : *Fichier › Enregistrer une copie dans GitHub*, message de commit : « Exploration : hypothèses H1 à H3 »